In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
all_requisitions = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling requisitions for: {project_name}")

    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.1/requisitions",
            headers=headers,
            params={
                "project_id": project_id,
                "page": page,
                "per_page": 100
            }
        )

        if response.status_code != 200:
            print(f"  Error {response.status_code}, skipping")
            break

        rows = response.json()

        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name

        all_requisitions.extend(rows)

        if len(rows) < 100:
            break

        page += 1
        time.sleep(0.3)

print(f"Done! Total requisitions: {len(all_requisitions)}")

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
import pandas as pd
import re

clean_rows = []
for row in all_requisitions:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

spark.sql("DROP TABLE IF EXISTS procore_requisitions_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_requisitions_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, , -1, Cancelled, , Cancelled, True)